In [0]:
%pip install sqlalchemy psycopg2-binary pyarrow scikit-learn matplotlib seaborn

In [0]:
import boto3
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sqlalchemy import create_engine

AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="secret-key")
RDS_HOST       = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER       = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS       = dbutils.secrets.get(scope="aws-credentials", key="rds-password")
BUCKET         = "enem-data-lake-gblzera"

s3 = boto3.client("s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="sa-east-1"
)
engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:5432/enem_db"
)
print("aqui foi 4")

In [0]:
print("carregando parquet do S3 silver...")
obj = s3.get_object(Bucket=BUCKET, Key="silver/enem/2022/enem_candidatos_projeto.parquet")
df  = pd.read_parquet(io.BytesIO(obj["Body"].read()))

print(f"{len(df):,} registros carregados")
print(f"  colunas: {df.columns.tolist()}")

In [0]:
# Selecionar apenas candidatos que fizeram todas as provas
NOTAS = ["nu_nota_cn", "nu_nota_ch", "nu_nota_lc", "nu_nota_mt", "nu_nota_redacao"]

df_km = df[NOTAS + ["sg_uf_residencia", "tp_escola", "q006"]].dropna(subset=NOTAS)
print(f"Candidatos com todas as notas: {len(df_km):,}")
print(f"  ({len(df_km)/len(df)*100:.1f}% do total)")

# Normalizar as notas (Z-score)
scaler  = StandardScaler()
X       = scaler.fit_transform(df_km[NOTAS])
print(f"\ndados normalizados, shape: {X.shape}")
print(f"  media pos normalizacao: {X.mean():.4f}")
print(f"  desvio pos normalizacaoo: {X.std():.4f}")

In [0]:
print("calculando inercia para k=2 a k=10...")
inercias = []
k_range  = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inercias.append(km.inertia_)
    print(f"  k={k} → inércia: {km.inertia_:,.0f}")

plt.figure(figsize=(9, 4))
plt.plot(list(k_range), inercias, marker="o", linewidth=2, color="#2563eb")
plt.xlabel("numero de clusters (k)")
plt.ylabel("inercia")
plt.title("elbow method, escolha do K ideal")
plt.xticks(list(k_range))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/elbow.png", dpi=120)
plt.show()
print("grafico salvo")

In [0]:
# K=4 representa bem 4 perfis naturais do ENEM:
# alto desempenho, médio-alto, médio-baixo, baixo desempenho
K = 4

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
df_km    = df_km.copy()
df_km["cluster"] = km_final.fit_predict(X)

# Métricas de qualidade
silhouette = silhouette_score(X, df_km["cluster"], sample_size=50_000, random_state=42)
db_score   = davies_bouldin_score(X, df_km["cluster"])

print(f" K-means aplicado com K={K}")
print(f"\nmetricas:")
print(f"  silhouette score : {silhouette:.4f}  (quanto mais próximo de 1, melhor)")
print(f"  davies-bouldin   : {db_score:.4f}   (quanto menor, melhor)")
print(f"  inercia          : {km_final.inertia_:,.0f}")
print(f"\ndistribuição dos clusters:")
print(df_km["cluster"].value_counts().sort_index())

In [0]:
# Médias reais por cluster (desnormalizadas)
resumo = df_km.groupby("cluster")[NOTAS + ["nu_media_total"] if "nu_media_total" in df_km.columns else NOTAS].mean()

# Renomear colunas para exibição
resumo_display = df_km.groupby("cluster")[NOTAS].mean().round(1)
resumo_display["total_candidatos"] = df_km.groupby("cluster").size()
resumo_display["pct"] = (resumo_display["total_candidatos"] / len(df_km) * 100).round(1)

print("medias por cluster:\n")
print(resumo_display.to_string())

# Nomear clusters por perfil
NOMES = {
    resumo_display["nu_nota_mt"].idxmax(): "Alto Desempenho",
    resumo_display["nu_nota_mt"].idxmin(): "Baixo Desempenho",
}
medios = [i for i in range(K) if i not in NOMES]
medias_medios = resumo_display.loc[medios, "nu_nota_mt"].sort_values(ascending=False)
NOMES[medias_medios.index[0]] = "medio-alto"
NOMES[medias_medios.index[1]] = "medio-baixo"

df_km["perfil"] = df_km["cluster"].map(NOMES)
print(f"\nperfis identificados: {NOMES}")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = ["#ef4444", "#f97316", "#3b82f6", "#22c55e"]

# Gráfico 1 — Média por cluster e disciplina
disciplinas = ["CN", "CH", "LC", "MT", "Redação"]
for i, (cluster_id, nome) in enumerate(NOMES.items()):
    medias = resumo_display.loc[cluster_id, NOTAS].values
    axes[0].plot(disciplinas, medias, marker="o", label=nome,
                 color=cores[i], linewidth=2)

axes[0].set_title("perfil de desempenho por cluster")
axes[0].set_ylabel("media das notas")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(300, 750)

# Gráfico 2 — Distribuição dos clusters
contagem = df_km["perfil"].value_counts()
axes[1].bar(contagem.index, contagem.values,
            color=[cores[list(NOMES.values()).index(n)] for n in contagem.index])
axes[1].set_title("numero de candidatos por perfil")
axes[1].set_ylabel("candidatos")
for i, v in enumerate(contagem.values):
    axes[1].text(i, v + 5000, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/clusters.png", dpi=120)
plt.show()
print("visualizacoes geradas")

In [0]:
# Salvar amostra dos clusters na REF
amostra = df_km[["cluster", "perfil"] + NOTAS + ["sg_uf_residencia", "tp_escola", "q006"]].copy()

amostra.to_sql(
    "enem_clusters",
    engine,
    schema="ref",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)
print(f"ref.enem_clusters — {len(amostra):,} registros salvos")

# Resumo final para apresentação
resumo_final = df_km.groupby("perfil").agg(
    total=("cluster", "count"),
    media_cn=("nu_nota_cn", "mean"),
    media_ch=("nu_nota_ch", "mean"),
    media_lc=("nu_nota_lc", "mean"),
    media_mt=("nu_nota_mt", "mean"),
    media_redacao=("nu_nota_redacao", "mean"),
).round(1).reset_index()

resumo_final.to_sql(
    "enem_clusters_resumo",
    engine,
    schema="ref",
    if_exists="replace",
    index=False
)
print(f"ref.enem_clusters_resumo — {len(resumo_final)} perfis")
print("\n=== RESULTADO FINAL ===")
print(resumo_final.to_string(index=False))
print(f"\nsilhouette score : {silhouette:.4f}")
print(f"davies-bouldin   : {db_score:.4f}")